In [ ]:
import os
import polars as pl
from src.utils.config import setup_clickhouse_client
from src.utils.helper import convert_row
from src.utils.spark import get_spark

spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

In [ ]:
# Fetch flights data
flights_query = "SELECT * FROM aviation_flights order by (dep_scheduled_time, dep_actual_time, arr_scheduled_time, arr_actual_time)"
flights_result = client.query(flights_query).result_rows
flights_columns = client.query(flights_query).column_names
flights_result = [convert_row(row) for row in flights_result]

# Create spark DataFrame
flights_pdf = pl.DataFrame(flights_result, schema=flights_columns, orient='row')

print(f"Total flight records retrieved: {len(flights_pdf)}")
flights_pdf.head(5)

In [ ]:
weather_query = "SELECT * FROM historical_weather_data order by date_observed"

weather_result = client.query(weather_query).result_rows
weather_result = [convert_row(row) for row in weather_result] 
weather_columns = client.query(weather_query).column_names

weather_pdf = pl.DataFrame(weather_result, schema=weather_columns, orient='row')

print(f"Total weather records retrieved: {len(weather_pdf)}")
weather_pdf.head(5)

In [ ]:
print('Prepared flights and weather data for merging.')

# Filter out cancelled and unknown status flights
valid_statuses = ['cancelled', 'unknown']
flights_filtered = flights_pdf.filter(~pl.col('status').is_in(valid_statuses)) \
                    .unique(subset=['flight_type', 'status', 'iata_number', 'airline_name', 
                                    'dep_scheduled_time', 'arr_scheduled_time', 'dep_actual_time', 'arr_actual_time'],
                                    keep='first')

print(f"Flight records after filtering: {len(flights_filtered)}")

In [ ]:
# Create hour columns for both departure and arrival times
flights_filtered = flights_filtered.with_columns(
    pl.col('dep_scheduled_time').str.to_datetime().dt.truncate('1h').alias('dep_hour'),
    pl.col('arr_scheduled_time').str.to_datetime().dt.truncate('1h').alias('arr_hour')
)

# Split into departure and arrival flights for conditional merging 
departure_flights = flights_filtered.filter(pl.col('flight_type') == 'departure')
arrival_flights = flights_filtered.filter(pl.col('flight_type') == 'arrival')

print(f"\nDeparture flights: {len(departure_flights)}, Arrival flights: {len(arrival_flights)}")
departure_flights.head(3)
arrival_flights.head(3)

In [ ]:
weather_pdf = weather_pdf.with_columns(
    pl.col('date_observed').str.to_datetime().dt.truncate('1h').alias('weather_hour') 
)

print(f"Weather records by hour: {len(weather_pdf)}")

weather_pdf.head(3)

In [ ]:
# Merge departure flights with weather
# Departure: join dep_hour with weather_hour
if len(departure_flights) > 0:
    merged_departures = departure_flights.join(
        weather_pdf,
        how='left',
        left_on='dep_hour',
        right_on='weather_hour',
        maintain_order='left'
    )

    merged_departures= merged_departures.drop(['arr_scheduled_time', 'arr_actual_time', 'arr_delay_mins']) \
        .sort(by=['dep_scheduled_time', 'dep_actual_time'], nulls_last=True) \
        .rename({'dep_scheduled_time': 'scheduled_time', 'dep_actual_time': 'actual_time', 'dep_delay_mins': 'delay_mins'})

    print(f"Merged departure flights: {len(merged_departures)}")
    merged_departures.head(3)

In [ ]:
# Merge arrival flights with weather
# Arrival: join arr_hour with weather_hour (same hour)
if len(arrival_flights) > 0:
    merged_arrivals = arrival_flights.join(
        weather_pdf,
        how='left',
        left_on='arr_hour',
        right_on='weather_hour',
        maintain_order='left'
    )
    
    merged_arrivals= merged_arrivals.drop(['dep_scheduled_time', 'dep_actual_time', 'dep_delay_mins']) \
    .sort(by=['arr_scheduled_time', 'arr_actual_time'], nulls_last=True) \
    .rename({'arr_scheduled_time': 'scheduled_time', 'arr_actual_time': 'actual_time', 'arr_delay_mins': 'delay_mins'})

    print(f"Merged arrival flights: {len(merged_arrivals)}")
    merged_arrivals.head(3)

In [ ]:
# Combine both datasets
if len(merged_departures) > 0 and len(merged_arrivals) > 0:
    combined_flights = pl.concat([merged_departures, merged_arrivals], how='vertical') \
            .sort(by=['scheduled_time', 'actual_time'], nulls_last=True)
    # print(combined_flights.columns)
    combined_flights = combined_flights.drop(['id', 'insert_time', 'iata_number', 'airline_name', 'codeshared_airline', 'codeshared_flight_number'])
else:
    combined_flights = pl.DataFrame()

print(f"\n Combined flights with weather data: {len(combined_flights)} rows")
combined_flights.head(3)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# Select features and target
feature_list = ["current_temp", "feels_like_temp", "pressure_hPa", "humidity_pct", "wind_speed_ms", "wind_deg", "cloudiness_pct", "rain_1h", "weather_main"]
features_var = combined_flights.select(feature_list)
independent_var = combined_flights["delay_mins"]

x_train, x_test, y_train, y_test = train_test_split(features_var.to_numpy(), independent_var.to_numpy(), test_size=0.2, random_state=42)

In [ ]:
#  Test multi-class problem with decision trees
#
#

y_statement = independent_var.to_numpy()
y_multiclass = np.select(
    [
        y_statement <= 15,
        (y_statement > 15) & (y_statement <= 30),
        (y_statement > 30) & (y_statement <= 45),
        y_statement > 45
    ],
    [0, 1, 2, 3]
)

class_counts = np.bincount(y_multiclass)

print(f"""
    0 (≤ 15 mins): {class_counts[0]}
    1 (15 - 30 mins): {class_counts[1]}
    2 (30 - 45 mins): {class_counts[2]}
    3 (> 45 mins): {class_counts[3]}
""")

features_var = features_var.with_columns(
    features_var["weather_main"].cast(str).rank("dense").cast(int).alias("weather_main")
)
results = []

X = combined_flights.select(features_var).to_numpy()
X = np.nan_to_num(X, nan=0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_multiclass, test_size=0.2, random_state=42
)

sample_weights = compute_sample_weight('balanced', y_train)
model = RandomForestClassifier(n_estimators=250, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

# Train on full train set for final evaluation
model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

results.append({
    'cv_mean_accuracy': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'test_accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1
})

# Convert to DataFrame and sort by test accuracy
results_df = pd.DataFrame(results).sort_values('test_accuracy', ascending=False)
print(results_df[['cv_mean_accuracy', 'cv_std', 'test_accuracy', 'precision', 'recall', 'f1_score']].head(5).to_string())

print("\n\nSummary:")
best = results_df.iloc[0]
print(f"Cross-Val Accuracy (mean±std): {best['cv_mean_accuracy']:.2%} ± {best['cv_std']:.2%}")
print(f"Test Accuracy: {best['test_accuracy']:.2%}")
print(f"Precision: {best['precision']:.2%}")
print(f"Recall: {best['recall']:.2%}")
print(f"f1_score: {best['f1_score']:.2%}")


In [ ]:
#  Test binary class problem with decision trees
#
#

y_statement = independent_var.to_numpy()
y_binary = (y_statement > 15).astype(int)

class_counts = np.bincount(y_binary)

print(f"Delays > 15 min: {class_counts[0]}, otherwise: {class_counts[1]}")

features_var = features_var.to_dummies(columns=["weather_main"])
results = []

X = combined_flights.select(features_var).to_numpy()
X = np.nan_to_num(X, nan=0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_multiclass, test_size=0.2, random_state=42
)

sample_weights = compute_sample_weight('balanced', y_train)
model = RandomForestClassifier(n_estimators=250, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

# Train on full train set for final evaluation
model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

results.append({
    'cv_mean_accuracy': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'test_accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1
})

# Convert to DataFrame and sort by test accuracy
results_df = pd.DataFrame(results).sort_values('test_accuracy', ascending=False)
print(results_df[['cv_mean_accuracy', 'cv_std', 'test_accuracy', 'precision', 'recall', 'f1_score']].head(5).to_string())

print("\n\nSummary:")
best = results_df.iloc[0]
print(f"Cross-Val Accuracy (mean±std): {best['cv_mean_accuracy']:.2%} ± {best['cv_std']:.2%}")
print(f"Test Accuracy: {best['test_accuracy']:.2%}")
print(f"Precision: {best['precision']:.2%}")
print(f"Recall: {best['recall']:.2%}")
print(f"f1_score: {best['f1_score']:.2%}")